<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em; margin-bottom:1em'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 2: nn.Module</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sys

assignment_path = '../../Assignment_1/'
sys.path.append(assignment_path)
from helpers import load_data

X, y = load_data(assignment_path + 'data.csv')
print(f'X: {X.shape},  y: {y.shape}')


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>nn.Linear — replacing manual yhat</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
In notebook 01 we computed the linear prediction manually:

```python
yhat = theta_1 * x + theta_0
```

`nn.Linear(in_features, out_features)` does exactly this — and automatically creates and manages the parameters.


In [ ]:
# same scalar input as notebook 01
x = torch.tensor([[2.0]])   # shape (1, 1) — one instance, one feature

linear = nn.Linear(in_features=1, out_features=1)
print(linear)


In [ ]:
# parameters are created automatically
print(f'weight (theta_1): {linear.weight}  shape={list(linear.weight.shape)}')
print(f'bias   (theta_0): {linear.bias}   shape={list(linear.bias.shape)}')
print(f'requires_grad:    {linear.weight.requires_grad}')


<!-- FULL: keep, DEMO: delete -->
Correspondence with our notation:

- `linear.weight` $\leftrightarrow$ $\theta_1$, shape `(out, in)`
- `linear.bias`   $\leftrightarrow$ $\theta_0$, shape `(out,)`

Both have `requires_grad=True` automatically — no need to set it manually.


In [ ]:
# forward pass — same as theta_1 * x + theta_0
yhat = linear(x)
print(f'yhat: {yhat}')

# verify manually
yhat_manual = linear.weight * x + linear.bias
print(f'manual: {yhat_manual}')
print(f'match:  {torch.allclose(yhat, yhat_manual)}')


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>nn loss functions</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch provides standard loss functions in `torch.nn`. For regression, `nn.MSELoss()` computes the mean squared error:

$$L = \\frac{1}{N} \\sum_{i=1}^N (\\hat{y}^{(i)} - y^{(i)})^2$$


In [ ]:
y_true = torch.tensor([[3.0]])

loss_fn = nn.MSELoss()
loss    = loss_fn(yhat, y_true)
print(f'loss: {loss.item():.4f}')


In [ ]:
# equivalent to our manual formula
loss_manual = ((yhat - y_true) ** 2).mean()
print(f'manual: {loss_manual.item():.4f}')
print(f'match:  {torch.allclose(loss, loss_manual)}')


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; nn vs functional losses</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Loss functions are also available in `torch.nn.functional` — stateless functions rather than objects:

```python
import torch.nn.functional as F
loss = F.mse_loss(yhat, y_true)   # identical result
```

`nn` style is more common in training loops; `F` style is convenient inside `forward()`.


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Training on real data — nn.Linear + manual SGD</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Now use `nn.Linear` and `nn.MSELoss` with the housing data. We keep manual SGD for now so the training loop stays familiar from notebook 01. The key difference: parameters are managed by the layer, not by hand.


In [ ]:
# single feature for clarity — same as notebook 01
X_train = X[:, 1:2]   # size feature, shape (N, 1)

torch.manual_seed(0)
linear  = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()
lr      = 0.1


In [ ]:
losses = []
for epoch in range(50):

    yhat = linear(X_train)              # forward
    loss = loss_fn(yhat, y)             # loss
    loss.backward()                     # backward

    with torch.no_grad():               # update
        for p in linear.parameters():
            p -= lr * p.grad

    linear.zero_grad()                  # zero gradients
    losses.append(loss.item())

print(f'Final loss: {losses[-1]:.4f}')
print(f'weight: {linear.weight.item():.4f},  bias: {linear.bias.item():.4f}')


<!-- FULL: keep, DEMO: delete -->
Note two improvements over notebook 01:

- `for p in linear.parameters()` — iterate over all parameters without naming them
- `linear.zero_grad()` — zeros all parameter gradients at once


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color='#005564', linewidth=2)
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('nn.Linear + manual SGD', color='#163C69', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Introducing ReLU</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`nn.ReLU()` applies the rectified linear unit element-wise:

$$a = \\max(0, z)$$

It is the most common activation function in modern networks. It introduces non-linearity, allowing the network to learn functions beyond simple linear mappings.


In [ ]:
relu = nn.ReLU()

z = torch.tensor([[-2.0, -1.0, 0.0, 1.0, 2.0]])
a = relu(z)
print(f'z: {z}')
print(f'a: {a}')


<!-- FULL: keep, DEMO: delete -->
A two-layer network applies: **Linear → ReLU → Linear**.
The hidden layer computes features; ReLU introduces non-linearity; the output layer combines them.

In [ ]:
# build manually — Linear -> ReLU -> Linear
layer1 = nn.Linear(in_features=4, out_features=8)
relu   = nn.ReLU()
layer2 = nn.Linear(in_features=8, out_features=1)

# forward pass
h1   = layer1(X)
h1_a = relu(h1)
yhat = layer2(h1_a)
print(f'h1   shape: {h1.shape}')
print(f'h1_a shape: {h1_a.shape}')
print(f'yhat shape: {yhat.shape}')


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>nn.Sequential</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`nn.Sequential` packages layers into a single object. Layers are applied in order during the forward pass — no need to write the chain manually.


In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
print(model)


In [ ]:
# all parameters accessible via model.parameters()
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params}')

for name, p in model.named_parameters():
    print(f'  {name:15s}  shape={list(p.shape)}')


In [ ]:
# forward pass — same as h1_a = relu(layer1(X)); yhat = layer2(h1_a)
yhat = model(X)
print(f'yhat shape: {yhat.shape}')


<!-- FULL: keep, DEMO: delete -->
Train with the same manual SGD loop:

In [ ]:
loss_fn = nn.MSELoss()
lr      = 1e-2
losses  = []

for epoch in range(100):
    loss = loss_fn(model(X), y)
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= lr * p.grad
    model.zero_grad()
    losses.append(loss.item())

print(f'Final loss: {losses[-1]:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color='#005564', linewidth=2)
ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss')
ax.set_title('nn.Sequential + manual SGD', color='#163C69', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; plain list instead of nn.Sequential / nn.ModuleList</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Layers stored in a plain Python list are **not** registered — `model.parameters()` will not see them and they will not be updated.


In [ ]:
class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)]  # plain list!

broken = BrokenModel()
print('Parameters:', list(broken.parameters()))   # empty!


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>nn.Module subclassing</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
`nn.Sequential` is convenient but limited — it always applies layers strictly in order with no branching or flexibility.

Subclassing `nn.Module` gives full control:

- custom `forward()` logic (skip connections, multiple inputs/outputs)
- configurable architecture (depth, width as constructor arguments)
- named submodules for easier inspection and debugging


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; forgetting super().__init__()</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Every `nn.Module` subclass must call `super().__init__()` first. Without it, parameter registration is not set up and you get a cryptic `AttributeError`.

In [ ]:
class BrokenModule(nn.Module):
    def __init__(self):
        # missing: super().__init__()
        self.linear = nn.Linear(4, 1)   # AttributeError!

BrokenModule()


<!-- FULL: keep, DEMO: delete -->
A correct minimal subclass:

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


In [ ]:
torch.manual_seed(0)
mlp = MLP(in_features=4, hidden=8, out_features=1)
print(mlp)
print(f'Parameters: {sum(p.numel() for p in mlp.parameters())}')


<!-- FULL: keep, DEMO: delete -->
`model.train()` and `model.eval()` switch between training and evaluation modes. Some layers (Dropout, BatchNorm — Session 7) behave differently in each mode. Always call `model.eval()` before validation.


In [ ]:
print(f'Default:       training={mlp.training}')
mlp.eval()
print(f'After eval():  training={mlp.training}')
mlp.train()
print(f'After train(): training={mlp.training}')


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Going Deeper</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
With `nn.Module` subclassing, depth and width become constructor arguments. The architecture is configurable without rewriting the class.


In [ ]:
class DeepMLP(nn.Module):
    """MLP with configurable depth and width."""

    def __init__(self, in_features, hidden_sizes, out_features):
        super().__init__()
        sizes  = [in_features] + hidden_sizes
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
shallow = DeepMLP(4, [8],       1)
deep    = DeepMLP(4, [32,16,8], 1)

print(f'Shallow: {sum(p.numel() for p in shallow.parameters())} parameters')
print(shallow)
print(f'\nDeep:    {sum(p.numel() for p in deep.parameters())} parameters')
print(deep)


<!-- FULL: keep, DEMO: delete -->
Train both and compare:

In [ ]:
def train(model, n_epochs=150, lr=1e-2):
    loss_fn = nn.MSELoss()
    losses  = []
    for _ in range(n_epochs):
        loss = loss_fn(model(X), y)
        loss.backward()
        with torch.no_grad():
            for p in model.parameters(): p -= lr * p.grad
        model.zero_grad()
        losses.append(loss.item())
    return losses


In [ ]:
torch.manual_seed(0)
losses_shallow = train(DeepMLP(4, [8],       1))
torch.manual_seed(0)
losses_deep    = train(DeepMLP(4, [32,16,8], 1))

print(f'Shallow final loss: {losses_shallow[-1]:.4f}')
print(f'Deep    final loss: {losses_deep[-1]:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses_shallow, color='#005564', linewidth=2, label='shallow [8]')
ax.plot(losses_deep,    color='#FF6A00', linewidth=2, label='deep [32,16,8]')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Shallow vs deep — training loss', color='#163C69', fontweight='bold')
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


<!-- FULL: keep, DEMO: delete -->
Note: we are training on the full dataset with no validation split. The training loss tells us how well the model fits the training data — not how well it generalises. Validation and overfitting are covered when we introduce `DataLoader` and the full training loop.
